# 11 — Evaluation and comparison


## Imports


In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import torch
import joblib
from torch import nn
import xgboost as xgb
from sklearn.metrics import f1_score


## Configuration


In [ ]:
from pathlib import Path
# ใช้ project root เดียวกัน ไม่ต้อง import โค้ด pipeline จาก program
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'dataset').is_dir() and (p / 'model').is_dir() and (p / 'program').is_dir())
RAW = ROOT / 'dataset/raw/mit-bih-arrhythmia-database-1.0.0'
DATA = ROOT / 'dataset/processed'
ARTIFACTS = ROOT / 'model/artifacts'
REPORTS = ROOT / 'model/reports'
CLASSES = ['N', 'S', 'V', 'F', 'Q']
SYMBOLS = ['NLRej', 'AaJS', 'VE', 'F', '/fQ']
LABEL_MAP = {s: i for i, group in enumerate(SYMBOLS) for s in group}
SEED = 42
PREPROCESSING = {'version': 1, 'fs': 360, 'left': 90, 'right': 162,
                 'band': [0.5, 40], 'order': 4, 'leads': ['MLII', 'V5']}

## Artifact helpers


In [ ]:
def read_json(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'ไม่พบ {path}; กรุณารัน notebook ขั้นก่อนหน้าก่อน')
    return json.loads(path.read_text(encoding='utf-8'))

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def new_stage(marker):
    # ห้ามเขียนทับผลเดิมเงียบ ๆ เพราะอาจทำให้โมเดลอ้างอิงคนละ split
    if Path(marker).exists():
        raise FileExistsError(f'มีผลขั้นนี้แล้ว: {marker}; โหลดผลเดิมแทนการรันซ้ำ')
    if (ARTIFACTS / 'selection.json').exists():
        raise RuntimeError('ล็อกโมเดลแล้ว ไม่อนุญาตให้เตรียมข้อมูลหรือ tuning ใหม่ใน experiment เดิม')

## Verify selection lock


In [ ]:
def verify_lock():
    lock = read_json(ARTIFACTS / 'selection.json')
    if lock['prepared_hash'] != sha256(DATA / 'prepared.json'):
        raise ValueError('Dataset manifest เปลี่ยนหลังล็อกโมเดล')
    for filename, field in [('preprocessing.json', 'preprocessing_hash'), ('imputer.joblib', 'imputer_hash')]:
        if sha256(DATA / filename) != lock[field]:
            raise ValueError('Preprocessing เปลี่ยนหลังล็อกโมเดล')
    for candidate in lock['candidates']:
        for filename, expected in candidate['files'].items():
            if sha256(ARTIFACTS / candidate['path'] / filename) != expected:
                raise ValueError(f'Checkpoint/artifact เปลี่ยนหลังล็อก: {filename}')
    return lock

## Load saved models


In [ ]:
class ECGNetwork(nn.Module):
    def __init__(self, kind, channels, kernel=7, hidden=64, dropout=.3):
        super().__init__()
        layers, previous = [], 1
        for width in channels:
            layers += [nn.Conv1d(previous, width, kernel, padding=kernel // 2),
                       nn.BatchNorm1d(width), nn.ReLU(), nn.MaxPool1d(2)]
            previous = width
        self.conv = nn.Sequential(*layers)
        self.kind = kind
        if kind == 'cnn_bilstm':
            self.lstm = nn.LSTM(previous, hidden, batch_first=True, bidirectional=True)
            previous = hidden * 2
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(previous, 5))

    def forward(self, x):
        x = self.conv(x)
        if self.kind == 'cnn_bilstm':
            _, (hidden, _) = self.lstm(x.transpose(1, 2))
            x = torch.cat([hidden[-2], hidden[-1]], dim=1)
        else:
            x = x.mean(dim=2)
        return self.head(x)

def network(kind, parameters):
    return ECGNetwork(kind, parameters['channels'], parameters.get('kernel', 7),
                      parameters.get('hidden', 64), parameters.get('dropout', .3))

def load_model(folder, info, device='cpu'):
    kind = info['kind']
    if kind in ['rf', 'dummy']:
        return joblib.load(folder / 'model.joblib')
    if kind == 'xgb':
        model = xgb.Booster()
        model.load_model(folder / 'model.ubj')
        return model
    model = network(kind, info['parameters'])
    model.load_state_dict(torch.load(folder / 'model.pt', map_location='cpu', weights_only=True))
    return model.to(device).eval()

def probabilities(model, kind, values, device='cpu', batch_size=128):
    if kind in ['rf', 'dummy']:
        output = np.zeros((len(values), 5), dtype=np.float32)
        output[:, np.asarray(model.classes_, dtype=int)] = model.predict_proba(values)
        return output
    if kind == 'xgb':
        return model.predict(xgb.DMatrix(values))
    outputs = []
    model.eval()
    with torch.inference_mode():
        for start in range(0, len(values), batch_size):
            # waveform: (beats, 252) -> Conv1d: (beats, 1, 252)
            batch = torch.as_tensor(values[start:start + batch_size, None, :], device=device)
            outputs.append(model(batch).softmax(dim=1).cpu().numpy())
    return np.concatenate(outputs)

def synchronize(device):
    if str(device).startswith('cuda'):
        torch.cuda.synchronize()

## Load partition helper


In [ ]:
def load_partition(split, *, allow_test=False):
    if split not in ['train', 'validation', 'test']:
        raise ValueError('Unknown partition')
    if split == 'test' and not allow_test:
        raise PermissionError('Test เปิดได้เฉพาะ final evaluation หลังล็อกโมเดล')
    manifest = read_json(DATA / 'prepared.json')
    folder = DATA / split
    for name in ['metadata.parquet', 'waveforms.npy', 'features.npy']:
        if sha256(folder / name) != manifest['files'][f'{split}/{name}']:
            raise ValueError(f'artifact เปลี่ยน: {split}/{name}')
    meta = pd.read_parquet(folder / 'metadata.parquet')
    waves, x = np.load(folder / 'waveforms.npy'), np.load(folder / 'features.npy')
    if len(meta) != len(waves) or len(meta) != len(x) or meta.beat_id.duplicated().any():
        raise ValueError('beat alignment ไม่ถูกต้อง')
    if not meta.split.eq(split).all() or set(meta.class_id) != set(range(5)):
        raise ValueError('partition หรือ class coverage ไม่ถูกต้อง')
    if not np.array_equal(meta.waveform_index.to_numpy(), np.arange(len(meta))):
        raise ValueError('waveform index ไม่ตรงกับ metadata')
    if waves.shape != (len(meta), 252) or not np.isfinite(waves).all() or not np.isfinite(x).all():
        raise ValueError('ขนาด waveform หรือค่าข้อมูลหลัง preprocessing ไม่ถูกต้อง')
    return {'meta': meta, 'wave': waves, 'x': x, 'y': meta.class_id.to_numpy()}

## Evaluation metrics


In [ ]:
def macro_f1(y, predicted):
    return float(f1_score(y, predicted, labels=list(range(5)), average='macro', zero_division=0))

def metrics(y, pred):
    report = classification_report(y, pred, labels=list(range(5)), target_names=CLASSES,
                                   zero_division=0, output_dict=True)
    matrix = confusion_matrix(y, pred, labels=list(range(5)))
    normalized = np.divide(matrix, matrix.sum(1, keepdims=True), out=np.zeros_like(matrix, dtype=float),
                           where=matrix.sum(1, keepdims=True) != 0)
    return {'accuracy': float(accuracy_score(y, pred)), 'macro_f1': macro_f1(y, pred),
            'balanced_accuracy': float(normalized.diagonal().mean()),
            'weighted_f1': float(report['weighted avg']['f1-score']), 'per_class': report,
            'confusion_matrix': matrix.tolist(), 'normalized_confusion_matrix': normalized.tolist()}

## Patient bootstrap


In [ ]:
def bootstrap_patients(frame, repeats=1000):
    rng = np.random.default_rng(SEED)
    groups = {group: part for group, part in frame.groupby('patient_group')}
    names = list(groups)
    scores, absent = [], 0
    for _ in range(repeats):
        # สุ่มทั้งผู้ป่วยและคง beats ทั้งกลุ่ม รวมกลุ่มซ้ำตาม bootstrap multiplicity
        sample = pd.concat([groups[name] for name in rng.choice(names, len(names), replace=True)])
        absent += sample.class_id.nunique() < 5
        scores.append(macro_f1(sample.class_id, sample.predicted))
    return {'lower_95': float(np.quantile(scores, .025)), 'upper_95': float(np.quantile(scores, .975)),
            'replicates': repeats, 'missing_class_fraction': absent / repeats,
            'fixed_labels': CLASSES, 'seed': SEED}

## Open Test after verifying lock


In [ ]:
lock = verify_lock()  # ต้องผ่านขั้นนี้ก่อนเปิด Test
target = REPORTS / 'final'
target.mkdir(parents=True, exist_ok=False)
test = load_partition('test', allow_test=True)
rows = []

## Evaluate and inspect errors


In [ ]:
for candidate in lock['candidates']:
    folder = ARTIFACTS / candidate['path']
    info = read_json(folder / 'info.json')
    model = load_model(folder, info, candidate['device'])
    values = test['wave'] if info['kind'] in ['cnn', 'cnn_bilstm'] else test['x']
    probabilities_ = probabilities(model, info['kind'], values, candidate['device'])
    pred = probabilities_.argmax(1)
    result = metrics(test['y'], pred)
    out = target / candidate['configuration']
    out.mkdir()
    frame = test['meta'].copy()
    frame['predicted'] = pred
    for i, name in enumerate(CLASSES):
        frame[f'score_{name}'] = probabilities_[:, i]
    frame.to_parquet(out / 'predictions.parquet', index=False)
    write_json(out / 'metrics.json', result)
    group_rows = []
    for column in ['record_id', 'patient_group', 'lead']:
        for group, subset in frame.groupby(column):
            group_rows.append({'group_type': column, 'group': str(group), 'beats': len(subset),
                               'classes_present': int(subset.class_id.nunique()),
                               'macro_f1_fixed_5': macro_f1(subset.class_id, subset.predicted)})
    pd.DataFrame(group_rows).to_csv(out / 'group_metrics.csv', index=False)
    errors = frame.loc[frame.class_id != frame.predicted]
    errors.to_csv(out / 'errors.csv', index=False)
    for key in ['confusion_matrix', 'normalized_confusion_matrix']:
        fig, ax = plt.subplots(figsize=(6, 5))
        plot = ax.imshow(result[key], cmap='Blues')
        ax.set(xticks=range(5), yticks=range(5), xticklabels=CLASSES, yticklabels=CLASSES,
               xlabel='Predicted', ylabel='True', title=candidate['configuration'])
        fig.colorbar(plot, ax=ax)
        fig.tight_layout()
        fig.savefig(out / f'{key}.png', dpi=160)
        plt.close(fig)
    # ตัวอย่างความผิดพลาดเลือกโดย deterministic ไม่เลือกเฉพาะภาพที่ดูดี
    if not errors.empty:
        example = errors.groupby(['class_id', 'predicted']).head(1).head(8)
        fig, axes = plt.subplots(len(example), 1, figsize=(9, 2 * len(example)), squeeze=False)
        for ax, (_, beat) in zip(axes[:, 0], example.iterrows()):
            ax.plot(test['wave'][int(beat.waveform_index)])
            ax.set_title(f"{beat.beat_id}: {CLASSES[int(beat.class_id)]} -> {CLASSES[int(beat.predicted)]}")
        fig.tight_layout()
        fig.savefig(out / 'error_examples.png', dpi=120)
        plt.close(fig)
    if candidate['configuration'] == lock['winner']:
        write_json(out / 'patient_bootstrap.json', bootstrap_patients(frame))
    row = {'configuration': candidate['configuration'], 'selected_winner': candidate['configuration'] == lock['winner'],
           'validation_macro_f1': candidate['validation_macro_f1'], 'baseline_validation_macro_f1': candidate['baseline_macro_f1'],
           'test_macro_f1': result['macro_f1'], 'accuracy': result['accuracy'],
           'balanced_accuracy': result['balanced_accuracy'], 'weighted_f1': result['weighted_f1'],
           'train_seconds': candidate['train_seconds'], 'tuning_seconds': candidate['tuning_seconds'],
           'model_bytes': sum(p.stat().st_size for p in folder.glob('model.*')), **candidate['latency']}
    for name in CLASSES:
        for metric in ['precision', 'recall', 'f1-score', 'support']:
            row[f'{name}_{metric}'] = result['per_class'][name][metric]
    rows.append(row)
    del model

## Save comparison and limitations


In [ ]:
comparison = pd.DataFrame(rows)
comparison.to_csv(target / 'comparison.csv', index=False)
write_json(target / 'evaluation.json', {'selection_hash': sha256(ARTIFACTS / 'selection.json'),
           'winner': lock['winner'], 'test_used_for_selection': False})
(target / 'limitations.md').write_text(
    '# ข้อจำกัด\n\n- ใช้ seed เดียวและผู้ป่วย Test จำนวนน้อย\n'
    '- คะแนนกลุ่มย่อยใช้ 5 labels คงที่ ต้องอ่านจำนวนคลาสที่มีจริงประกอบ\n'
    '- ML ใช้ RR และ morphology; DL ใช้ waveform\n'
    '- ใช้ annotation centers และ offline filtering ไม่ได้ประเมิน R-peak detector\n'
    '- ผู้ชนะเลือกจาก Validation เท่านั้น ไม่เปลี่ยนตาม Test\n', encoding='utf-8')
comparison